# FDTDX Colab GPU Smoke Test

Run this notebook on a Colab GPU runtime. The key success signal is that JAX reports a `CudaDevice`, then FDTDX completes a tiny `place_objects -> apply_params -> run_fdtd` simulation.

In [ ]:
import os
import platform
import shutil
import sys

print('python:', sys.version)
print('platform:', platform.platform())
print('COLAB_GPU:', os.environ.get('COLAB_GPU'))
print('nvidia-smi:', shutil.which('nvidia-smi'))

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('No NVIDIA GPU runtime detected. In Colab, choose Runtime > Change runtime type > GPU.')

!nvidia-smi

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('fdtdx') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-qU', 'fdtdx[cuda12]'])
else:
    print('fdtdx already installed')

print('If pip changed numpy/jax in this runtime, restart the runtime once before running the next cells.')

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import fdtdx

print('numpy:', np.__version__)
print('jax:', jax.__version__)
print('devices:', jax.devices())
print('backend:', jax.default_backend())

if jax.default_backend() != 'gpu':
    raise RuntimeError('JAX is not using the GPU backend.')

x = jnp.linspace(0, 1, 1024 * 1024)
y = jax.jit(lambda a: jnp.sum(jnp.sin(a) ** 2))(x)
print('jit_result:', float(y))

In [ ]:
import time

key = jax.random.PRNGKey(0)
config = fdtdx.SimulationConfig(
    time=2e-15,
    resolution=200e-9,
    backend='gpu',
    dtype=jnp.float32,
)
print('steps:', config.time_steps_total, 'backend:', config.backend)

volume = fdtdx.SimulationVolume(
    partial_real_shape=(1.0e-6, 1.0e-6, 1.0e-6),
    material=fdtdx.Material(permittivity=1.0),
)
bound_cfg = fdtdx.BoundaryConfig.from_uniform_bound(boundary_type='periodic')
bound_dict, constraints = fdtdx.boundary_objects_from_config(bound_cfg, volume)
object_list = [volume, *bound_dict.values()]

source = fdtdx.GaussianPlaneSource(
    partial_grid_shape=(None, None, 1),
    partial_real_shape=(0.6e-6, 0.6e-6, None),
    fixed_E_polarization_vector=(1, 0, 0),
    wave_character=fdtdx.WaveCharacter(wavelength=1.55e-6),
    radius=0.25e-6,
    std=1 / 3,
    direction='-',
)
constraints.append(
    source.place_relative_to(
        volume,
        axes=(0, 1, 2),
        own_positions=(0, 0, 0),
        other_positions=(0, 0, 0),
    )
)
object_list.append(source)

energy = fdtdx.EnergyDetector(
    name='energy_last',
    as_slices=False,
    switch=fdtdx.OnOffSwitch(fixed_on_time_steps=[-1]),
)
constraints.extend(energy.same_position_and_size(volume))
object_list.append(energy)

key, subkey = jax.random.split(key)
objects, arrays, params, config, _ = fdtdx.place_objects(
    object_list=object_list,
    config=config,
    constraints=constraints,
    key=subkey,
)
arrays, objects, info = fdtdx.apply_params(arrays, objects, params, key)

start = time.time()
_, arrays = fdtdx.run_fdtd(
    arrays=arrays,
    objects=objects,
    config=config,
    key=key,
    show_progress=False,
)
runtime = time.time() - start

energy_sum = jnp.sum(arrays.detector_states['energy_last']['energy'])
print('runtime_s:', runtime)
print('detectors:', arrays.detector_states.keys())
print('energy_sum:', float(energy_sum))
print('smoke_ok:', bool(jnp.isfinite(energy_sum)))